# Example 15.6 / 3: Grow a Model One Prompt at a Time

Companion notebook for **15.6 Talks on AI for Optimization** in the AMPL lab Documentation, section "Grow a Model One Prompt at a Time". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Grow a Model One Prompt at a Time

Each answer should keep the previous model's meaning while changing one thing. In the stochastic
version, scenarios index the uncertain demand and the shipments, while the decision to open a
facility stays unindexed, because it is made before demand is known
(Two-Stage Recourse and Nonanticipativity (AMPL lab documentation, section 12.6.3)). The expected cost weights each
scenario's shipping cost by its probability. A small instance of the model to expect:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FACILITIES;                                  # candidate sites
set CUSTOMERS;                                   # demand points
set SCENARIOS;                                   # demand scenarios
param prob {SCENARIOS} > 0;                      # scenario probabilities
check: abs(sum {s in SCENARIOS} prob[s] - 1) < 1e-9;
param open_cost {FACILITIES} >= 0;               # dollars to open a site
param capacity {FACILITIES} >= 0;                # units a site can ship
param ship_cost {FACILITIES, CUSTOMERS} >= 0;    # dollars per unit shipped
param demand {CUSTOMERS, SCENARIOS} >= 0;        # units required

var Open {FACILITIES} binary;                    # first stage: open before demand is known
var Ship {FACILITIES, CUSTOMERS, SCENARIOS} >= 0; # second stage: shipments in each scenario

minimize Expected_Cost:
    sum {f in FACILITIES} open_cost[f] * Open[f]
  + sum {s in SCENARIOS} prob[s] *
        sum {f in FACILITIES, c in CUSTOMERS} ship_cost[f,c] * Ship[f,c,s];

subject to Meet_Demand {c in CUSTOMERS, s in SCENARIOS}:
    sum {f in FACILITIES} Ship[f,c,s] = demand[c,s];            # every scenario's demand met
subject to Capacity {f in FACILITIES, s in SCENARIOS}:
    sum {c in CUSTOMERS} Ship[f,c,s] <= capacity[f] * Open[f];  # only open sites ship

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

ampl.set["FACILITIES"] = ["Albany", "Syracuse", "Buffalo"]
ampl.set["CUSTOMERS"] = ["NYC", "Rochester", "Ithaca"]
ampl.set["SCENARIOS"] = ["low", "high"]
ampl.param["prob"] = {"low": 0.5, "high": 0.5}
ampl.param["open_cost"] = {"Albany": 500, "Syracuse": 400, "Buffalo": 300}
ampl.param["capacity"] = {"Albany": 150, "Syracuse": 150, "Buffalo": 150}
ampl.param["ship_cost"] = pd.DataFrame(
    [[2, 5, 4], [4, 2, 1], [6, 1, 3]],
    index=["Albany", "Syracuse", "Buffalo"], columns=["NYC", "Rochester", "Ithaca"],
)
ampl.param["demand"] = pd.DataFrame(
    {"low": [80, 30, 20], "high": [140, 50, 40]}, index=["NYC", "Rochester", "Ithaca"],
)

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, round(ampl.obj["Expected_Cost"].value(), 2))
print("open:", [f for f, v in ampl.var["Open"].to_dict().items() if v > 0.5])

HiGHS 1.14.0: optimal solution; objective 1150
35 simplex iterations
0 branching nodes
solved 1150.0
open: ['Albany', 'Buffalo']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set FACILITIES := Albany Syracuse Buffalo;
set CUSTOMERS := NYC Rochester Ithaca;
set SCENARIOS := low high;
param prob := low 0.5  high 0.5;
param:     open_cost  capacity :=
Albany          500       150
Syracuse        400       150
Buffalo         300       150;
param ship_cost:  NYC  Rochester  Ithaca :=
Albany              2          5       4
Syracuse            4          2       1
Buffalo             6          1       3;
param demand:  low  high :=
NYC             80   140
Rochester       30    50
Ithaca          20    40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Expected_Cost, Open;

HiGHS 1.14.0: optimal solution; objective 1150
35 simplex iterations
0 branching nodes
Expected_Cost = 1150

Open [*] :=
  Albany  1
 Buffalo  1
Syracuse  0
;

